# Phase 5-6: Embedding, ChromaDB, BM25

Phase 1-4 sudah selesai di preprocessing/finalizer. Notebook ini mulai dari chunk final `processed_chunks_ringan_pasal_chroma_ready.json`.


## Mapping Fase ke Implementasi Saat Ini

Pipeline produksi di `pipeline_legal_rag_indonesia.md` mendefinisikan Phase 1-11. Implementasi notebook ini memakai varian praktis ChromaDB:

- Phase 1-3: dikerjakan oleh `praproses_ringan_pasal.py`.
- Phase 4: dikerjakan oleh `finalize_chunks_for_chroma.py`, menghasilkan `../data/processed_chunks_ringan_pasal_chroma_ready.json`.
- Phase 5: embedding memakai `embedding_text` dengan `intfloat/multilingual-e5-base`.
- Phase 6: vector store memakai ChromaDB lokal sesuai proposal final.
- Phase 7: hybrid retrieval memakai dense Chroma + BM25 lokal.
- Phase 8: reranker dibuat opsional. Default mati supaya notebook ringan.
- Phase 9: context assembler memakai `display_text` + `citation_text`; sibling expansion bisa ditambah setelah chunk ayat/sibling tersedia stabil.
- Phase 10: generation wajib menyebut sumber hukum secara natural, misalnya `Peraturan Pemerintah No. 35 Tahun 2021, Pasal 52`, tanpa ID `[R#]` di jawaban.
- Phase 11: evaluasi inference + hook RAGAS.


In [ ]:
import hashlib
import json
import os
import pickle
import re
import shutil
from pathlib import Path
from typing import Dict, List, Any

import chromadb
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

BASE_DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
TYPO_CORRECTED_DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready_typo_corrected.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Merge typo-corrected artifact into the default Chroma-ready path by default.
# This keeps downstream notebooks/scripts using the canonical file name.
if TYPO_CORRECTED_DATA_PATH.exists():
    backup_path = BASE_DATA_PATH.with_suffix(".before_typo_corrected.json")
    if BASE_DATA_PATH.exists() and not backup_path.exists():
        shutil.copy2(BASE_DATA_PATH, backup_path)
        print(f"Backup base chunks: {backup_path}")
    shutil.copy2(TYPO_CORRECTED_DATA_PATH, BASE_DATA_PATH)
    print(f"Merged typo-corrected chunks into: {BASE_DATA_PATH}")
else:
    print("Typo-corrected chunks not found; using existing base Chroma-ready chunks.")

DATA_PATH = BASE_DATA_PATH

print(f"Device: {DEVICE}")
print(f"Data: {DATA_PATH}")
print(f"Chroma: {CHROMA_DB_DIR} / {COLLECTION_NAME}")


In [ ]:
def load_chunks(path: Path = DATA_PATH) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"{path} tidak ditemukan. Jalankan praproses_ringan_pasal.py lalu finalize_chunks_for_chroma.py dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    required = {"id", "text", "display_text", "embedding_text", "citation_text", "metadata"}
    missing = [i for i, c in enumerate(chunks[:20]) if not required.issubset(c)]
    if missing:
        raise ValueError(f"Chunk belum pakai schema baru. Cek index sample: {missing}")
    return ensure_unique_chunk_ids(chunks)


def ensure_unique_chunk_ids(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = {}
    fixed = 0
    for idx, chunk in enumerate(chunks):
        base_id = str(chunk.get("id") or f"chunk-{idx}")
        count = seen.get(base_id, 0)
        seen[base_id] = count + 1
        if count:
            seed = "::".join([
                base_id,
                str(idx),
                chunk.get("metadata", {}).get("source_file", ""),
                chunk.get("metadata", {}).get("pasal_id", ""),
                chunk.get("text", "")[:200],
            ])
            chunk["original_id"] = base_id
            chunk["id"] = hashlib.sha1(seed.encode("utf-8")).hexdigest()
            fixed += 1
    if fixed:
        print(f"Fixed duplicated chunk ids in memory: {fixed}")
    return chunks


def normalize_metadata(chunk: Dict[str, Any]) -> Dict[str, Any]:
    meta = dict(chunk.get("metadata", {}))
    meta["chunk_id"] = chunk.get("id", "")
    meta["citation_text"] = chunk.get("citation_text", "")
    meta["source_file"] = meta.get("source_file", "")
    meta["pasal_id"] = meta.get("pasal_id", "")
    meta["bab"] = meta.get("bab", "")
    meta["bab_title"] = meta.get("bab_title", "")
    meta["regulation_type"] = meta.get("regulation_type", "")
    meta["nomor"] = meta.get("nomor", "")
    meta["tentang"] = meta.get("tentang", "")
    meta["year"] = int(meta.get("year") or 0)
    meta["publication_year"] = int(meta.get("publication_year") or meta.get("year") or 0)
    safe = {}
    for key, value in meta.items():
        if value is None:
            safe[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            safe[key] = value
        else:
            safe[key] = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    return safe


def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9]+", text.lower())


def compact_citation(meta: Dict[str, Any]) -> str:
    reg_type = str(meta.get("regulation_type") or "Aturan").strip()
    nomor = str(meta.get("nomor") or "").strip()
    year = str(meta.get("publication_year") or meta.get("year") or "").strip()
    pasal = str(meta.get("pasal_id") or "").strip()

    parts = [reg_type]
    if nomor and nomor.lower() != "unknown":
        parts.append(f"No. {nomor}")
    if year and year != "0":
        parts.append(f"Tahun {year}")
    citation = " ".join(parts).strip()
    if pasal:
        citation = f"{citation}, {pasal}"
    return citation


def build_reference(meta: Dict[str, Any], idx: int | None = None) -> str:
    # idx dipertahankan untuk kompatibilitas cell lama, tapi output tetap sitasi natural.
    return compact_citation(meta)

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from {DATA_PATH}")
print("Sample citation:", chunks[0]["citation_text"])

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))

# Rebuild collection supaya tidak kecampur schema lama.
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted old collection: {COLLECTION_NAME}")
except Exception:
    pass

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine", "schema": "ringan_pasal_chroma_ready_v1"},
)

batch_size = 64 if DEVICE == "cuda" else 32
for start in tqdm(range(0, len(chunks), batch_size), desc="Embedding + storing"):
    batch = chunks[start:start + batch_size]
    ids = [c["id"] for c in batch]
    passages = ["passage: " + c["embedding_text"] for c in batch]
    embeddings = embedding_model.encode(
        passages,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).tolist()
    documents = [c["display_text"] for c in batch]
    metadatas = [normalize_metadata(c) for c in batch]
    collection.add(ids=ids, embeddings=embeddings, documents=documents, metadatas=metadatas)

print("Chroma count:", collection.count())

In [ ]:
bm25_corpus = [tokenize_for_bm25(c["display_text"] + " " + c["embedding_text"]) for c in chunks]
bm25 = BM25Okapi(bm25_corpus)
bm25_ids = [c["id"] for c in chunks]
BM25_PATH.parent.mkdir(parents=True, exist_ok=True)
with BM25_PATH.open("wb") as f:
    pickle.dump({"bm25": bm25, "ids": bm25_ids}, f)
print(f"BM25 saved to {BM25_PATH} ({len(bm25_ids)} docs)")

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload["bm25"]
    bm25_ids = payload["ids"]
    print(f"BM25 loaded: {len(bm25_ids)} docs")
else:
    print("BM25 index tidak ditemukan. Retrieval tetap jalan dengan dense Chroma saja.")


def dense_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].tolist()
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"],
    )
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 30) -> List[Dict[str, Any]]:
    if bm25 is None:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights: List[float] | None = None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = item["score"]
        out.append(hit)
    return out


def lex_posterior_score(hit: Dict[str, Any]) -> float:
    meta = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))

    try:
        year = int(meta.get("publication_year") or meta.get("year") or 0)
    except Exception:
        year = 0
    try:
        hierarchy = int(meta.get("regulation_hierarchy") or 99)
    except Exception:
        hierarchy = 99

    if str(meta.get("active_status", "")).lower() == "berlaku":
        score += 0.030
    score += min(max(year - 2000, 0), 40) * 0.001
    score += max(0, 6 - hierarchy) * 0.003

    # Jangan buang OCR/noisy docs karena bisa saja dokumen penting seperti PP 35/2021.
    # Cukup penalti ringan supaya dokumen bersih naik jika relevansinya mirip.
    if meta.get("quality_status") == "needs_review":
        score -= 0.010

    return score


def dedupe_legal_hits(hits: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    ranked = sorted(hits, key=lex_posterior_score, reverse=True)
    seen = set()
    out = []
    for hit in ranked:
        meta = hit.get("metadata", {})
        key = (
            meta.get("source_file", ""),
            meta.get("pasal_id", ""),
            meta.get("chunk_kind", ""),
            meta.get("chunk_index", ""),
        )
        if key in seen:
            continue
        seen.add(key)
        hit["final_score"] = lex_posterior_score(hit)
        out.append(hit)
        if len(out) >= k:
            break
    return out

def retrieve_documents(query: str, k: int = 6, fetch_k: int = 30, use_bm25: bool = True) -> List[Dict[str, Any]]:
    dense_hits = dense_search(query, fetch_k=fetch_k)
    sparse_hits = bm25_search(query, fetch_k=fetch_k) if use_bm25 else []
    fused = rrf_fuse([dense_hits, sparse_hits], weights=[1.0, 0.7]) if sparse_hits else dense_hits
    return dedupe_legal_hits(fused, k=k)


def print_references(docs: List[Dict[str, Any]]) -> None:
    seen = set()
    for doc in docs:
        citation = compact_citation(doc["metadata"])
        if citation in seen:
            continue
        seen.add(citation)
        print(f"- {citation}")

# Smoke test
smoke_docs = retrieve_documents("berapa pesangon pekerja yang di PHK", k=5)
print_references(smoke_docs)